# 현실의 데이터 10: 법인카드 지출 관리

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SuminHan/book-ml/blob/main/notebooks/datashop/10_corporate_card.ipynb)

**분야**: 금융 · 핀테크 &nbsp;|&nbsp; **이 분야에서 먼저 보는 문제**: 지출 관리 (spend management) · 비용 자동 분류 · 이상 지출 찾기

> 공공기관 법인카드 결제 44만 건. 가맹점 이름만 보고 무슨 비용인지 자동으로 나눌 수 있을까?

**데이터**: Oklahoma Purchase Card (P-card) FY2014 (약 80MB). 미국 오클라호마 주정부 공개 데이터: 주 정부 기관 · 대학 124곳의 2013-07 ~ 2014-06 법인카드 결제 442,458건 (기관 · 적요 · 금액 · 가맹점 · 날짜 · 가맹점 업종 코드 MCC). 카드 소지자 이름 열은 노트북에서 바로 지운다.

인턴 · 산학협력 과제에서 처음 받게 될 데이터와 닮은 공개 데이터다. 데이터를 열어 보고, 아래 예시처럼 문제 하나를 정해 기준선과 비교한 뒤, 맨 끝의 **나의 다섯 칸** 을 채운다 (1주 2차시 '문제 정형화와 평가지표').

| 칸 | 이 노트북의 예시 |
|---|---|
| ① 한 행 | 법인카드 결제 한 건 |
| ② 입력 X | 가맹점 이름 (글자) |
| ③ 출력 y | 비용 항목 (사무용품 · 출장 · 장비 · 통신 …) |
| ④ 평가지표 | 정확도 + 항목별 재현율 |
| ⑤ 기준선 | 가장 흔한 항목 |

**논문 · 출처** (직접 열어 보자: 이 데이터를 누가, 왜, 어떻게 만들었나):
- [Oklahoma Open Data: Purchase Card (PCard) Fiscal Year 2014](https://data.ok.gov/dataset/purchase-card-pcard-fiscal-year-2014)

In [1]:
import io, os, zipfile, urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def download(url, path):
    """Download url to path once (skip if it already exists)"""
    os.makedirs(os.path.dirname(path) or ".", exist_ok=True)
    if not os.path.exists(path):
        req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0 (ML course data shop)"})
        with urllib.request.urlopen(req, timeout=600) as r, open(path, "wb") as f:
            f.write(r.read())
    return path

## 입장: 데이터 받기

약 80MB. 사람 이름 (카드 소지자) 은 분석에 필요 없으니 받자마자 지운다 (개인정보 최소화).

In [2]:
URL = ("https://data.ok.gov/dataset/5f1e52fa-670d-4bb6-9072-4801e8c9ba17/resource/506a16ef-fcb6-4b03-8122-57cec56eb1b9"
       "/download/data-purchase-card-pcard-fiscal-year-2014.csv")
df = pd.read_csv(download(URL, "data/ok_pcard_2014.csv"), low_memory=False)
df = df.drop(columns=["Cardholder Last Name", "Cardholder First Initial"])
df = df.rename(columns={"Agency Name": "agency", "Description": "memo", "Amount": "amount", "Vendor": "vendor",
                        "Transaction Date": "date", "Merchant Category Code (MCC)": "mcc"})
df["date"] = pd.to_datetime(df["date"], format="%m/%d/%Y %I:%M:%S %p")
print(df.shape, df["date"].min().date(), "~", df["date"].max().date())
df[["agency", "memo", "amount", "vendor", "date", "mcc"]].sample(6, random_state=0)

(442458, 9) 2013-04-17 ~ 2014-06-30


,agency,memo,amount,vendor,date,mcc
216735,UNIV. OF OKLA. HEALTH SCIENCES CENTER,SEE INVOICE # YXMP0601 EA,123.54,ADI SECURITY PRD OK,2013-09-17,ELECTRICAL PARTS AND EQUIPMENT
52163,OKLAHOMA STATE UNIVERSITY,GENERAL PURCHASE,749.50,BLS Conduit,2013-12-12,COMPUTER SOFTWARE STORES
361073,OKLA. CAREER AND TECHNOLOGY EDUCATION,ROOM CHARGES,154.00,COMFORT INN & SUITES,2014-03-25,COMFORT HOTEL INTERNATIONAL
384701,OKLAHOMA STATE UNIVERSITY,7 1/8X4.5X13.75 #12 GROCER BD,74.83,ULINE SHIP SUPPLIES,2014-06-04,CATALOG MERCHANTS
131692,OKLA. PANHANDLE STATE UNIV.,Medical Terminology for He PCE,13.16,AMAZON MKTPLACE PMTS,2014-01-16,BOOK STORES
215203,UNIV. OF OKLA. HEALTH SCIENCES CENTER,GENERAL PURCHASE,11.97,WAL-MART #0269,2013-08-27,"GROCERY STORES,AND SUPERMARKETS"


## 열어 보기

금액 분포 (음수 = 환불), 업종 코드 (MCC) 가 몇 가지일까? 435가지는 너무 잘다. 회계 담당자가 쓰는 **비용 항목 10개 정도** 로 묶어 보자 (규칙은 업종 이름의 단어).

In [3]:
print(df["amount"].describe().round(1).to_dict())
print(f"refunds (negative): {(df['amount'] < 0).mean():.1%} | distinct MCC: {df['mcc'].nunique()}")
RULES = [("travel", ["AIRLINE", "AIR CARRIER", "HOTEL", "MOTEL", "LODGING", "TRAVEL AGENC", "CAR RENTAL", "MARRIOTT", "HILTON", "HOLIDAY INN", "SHERATON", "HYATT", "BEST WESTERN", "TAXI", "PARKING"]),
         ("office & books", ["STATIONERY", "OFFICE", "BOOK", "PRINTING", "NEWSPAPER", "PERIODICAL"]),
         ("food", ["GROCERY", "RESTAURANT", "EATING", "FAST FOOD", "CATERER", "BAKER", "DAIRY", "CANDY"]),
         ("medical & lab", ["MEDICAL", "DENTAL", "DRUG", "OPHTHALMIC", "HOSPITAL", "LABORATOR"]),
         ("IT & electronics", ["COMPUTER", "SOFTWARE", "ELECTRONIC", "TELECOMMUNICATION", "CABLE", "TELEPHONE"]),
         ("tools & facilities", ["HARDWARE", "INDUSTRIAL", "PLUMBING", "ELECTRICAL", "HOME SUPPLY", "LUMBER", "PAINT", "EQUIPMENT", "AUTOMOTIVE", "TIRE"]),
         ("shipping & services", ["COURIER", "POSTAL", "BUSINESS SERVICE", "PROFESSIONAL SERVICE", "ADVERTISING", "GOVERNMENT SERVICE", "CLEANING"]),
         ("retail & catalog", ["RETAIL", "CATALOG", "DIRECT MARKET", "DIRCT MARKET", "DEPARTMENT STORE", "DISCOUNT STORE", "WHOLESALE"])]

def expense_group(mcc):
    m = str(mcc).upper()
    for name, words in RULES:
        if any(w in m for w in words):
            return name
    return "other"

df["group"] = df["mcc"].map(expense_group)
print(df["group"].value_counts(normalize=True).round(3).to_dict())

{'count': 442458.0, 'mean': 425.0, 'std': 5266.5, 'min': -42863.0, '25%': 30.9, '50%': 104.9, '75%': 345.0, 'max': 1903858.4}
refunds (negative): 3.3% | distinct MCC: 435


{'other': 0.208, 'tools & facilities': 0.171, 'office & books': 0.155, 'travel': 0.121, 'food': 0.078, 'shipping & services': 0.075, 'IT & electronics': 0.072, 'retail & catalog': 0.066, 'medical & lab': 0.054}


## 문제 하나 골라 보기: 가맹점 이름 → 비용 항목

지출 관리 서비스는 결제가 들어오면 **비용 항목을 자동으로** 붙여 준다. 가맹점 이름의 글자 조각 (2~4글자) 으로 로지스틱회귀.
**시간 순서**: 2014년 3월까지 학습, 4~6월 시험 (실제로는 미래의 결제를 분류하니까). 시간을 줄이려고 학습은 15만 건만.

In [4]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline

tr = df[df["date"] < "2014-04-01"].sample(150000, random_state=0)
te = df[df["date"] >= "2014-04-01"]
model = make_pipeline(TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 4), min_df=3, max_features=200000),
                      LogisticRegression(C=3, max_iter=2000)).fit(tr["vendor"].astype(str), tr["group"])
pred = model.predict(te["vendor"].astype(str))
most = tr["group"].value_counts().idxmax()
print(f"baseline (always '{most}'): accuracy {np.mean(te['group'] == most):.1%}")
print(f"vendor-name character n-grams: accuracy {np.mean(te['group'] == pred):.1%}")
print("recall per group:", pd.Series(te["group"].values == pred).groupby(te["group"].values).mean().round(2).to_dict())
new_vendor = ~te["vendor"].isin(set(tr["vendor"]))
print(f"vendors never seen in training: {new_vendor.mean():.0%} of test, accuracy on them {np.mean(te['group'][new_vendor] == pred[new_vendor]):.1%}")

baseline (always 'other'): accuracy 21.2%
vendor-name character n-grams: accuracy 92.2%
recall per group: {'IT & electronics': 0.83, 'food': 0.94, 'medical & lab': 0.91, 'office & books': 0.95, 'other': 0.93, 'retail & catalog': 0.86, 'shipping & services': 0.85, 'tools & facilities': 0.94, 'travel': 0.98}
vendors never seen in training: 23% of test, accuracy on them 81.7%


### 이상 지출 찾기: 규칙으로 먼저

라벨이 없는 문제다. 감사 담당자가 먼저 보는 규칙 몇 가지를 세어 보자.
- 주말 결제
- 같은 카드 소지자 · 같은 가맹점 · 같은 날 여러 번 (한도를 피하려는 **쪼개기 결제** 의심) — 여기서는 이름을 지웠으니 '같은 기관 · 같은 가맹점 · 같은 날 · 같은 금액' 으로 센다
- 정확히 딱 떨어지는 큰 금액 (예: 5,000.00)

In [5]:
df["weekend"] = df["date"].dt.dayofweek >= 5
dup = df.groupby(["agency", "vendor", "date", "amount"]).size()
print(f"weekend payments: {df['weekend'].mean():.1%}")
print(f"(agency, vendor, day, amount) repeated 3+ times: {(dup >= 3).sum():,} groups, {dup[dup >= 3].sum():,} payments")
print(f"round amounts >= 1,000: {((df['amount'] >= 1000) & (df['amount'] % 100 == 0)).sum():,}")
print(df[df["weekend"]].groupby("group").size().sort_values(ascending=False).head(5).to_dict())

weekend payments: 8.3%
(agency, vendor, day, amount) repeated 3+ times: 3,465 groups, 15,736 payments
round amounts >= 1,000: 1,756
{'office & books': 7929, 'other': 6724, 'IT & electronics': 4250, 'food': 3737, 'travel': 3726}


**관찰**: 가맹점 이름만으로도 비용 항목을 꽤 맞히지만, **처음 보는 가맹점** 에서는 떨어진다. 실제 서비스는 가맹점 업종 코드 · 금액 · 적요 · 영수증 사진까지 함께 쓴다. 규칙으로 찾은 '의심 결제' 는 대부분 정상이다 (주말 출장 등). 그래서 이상 탐지의 지표는 '담당자가 하루에 검토할 수 있는 건수 중 진짜 문제의 비율' 이 된다.

## 다른 문제로 바꿔 보기

- **예산 예측**: 한 행 = 기관 × 달, 출력 = 다음 달 카드 지출 총액.
- **영수증 읽기**: 영수증 사진 → 가맹점 · 금액 · 날짜 (문서 이해 노트북과 연결).

### 나의 다섯 칸

아래 표를 직접 채워 보자. 위의 예시와 다르게 정해도 된다.

| 칸 | 내가 정한 것 |
|---|---|
| ① 한 행은 무엇? | |
| ② 입력 X (예측할 때 알 수 있는 것만) | |
| ③ 출력 y | |
| ④ 평가지표 | |
| ⑤ 기준선 | |